# 01. Environment & Repo Setup
# 02. Data Loader, Alignment Checks, Metric

Run each cell in order. If a step fails, re-run the cell after fixing the issue.

In [19]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [20]:
# !pip uninstall -y paddleocr paddlepaddle paddlepaddle-gpu

# # In your Colab before running the script
!pip install "paddlepaddle-gpu==2.6.2" "paddleocr==2.9.0" "jiwer==3.0.3"

# %cd /content
# !git clone https://github.com/PaddlePaddle/PaddleOCR.git
# %cd PaddleOCR

# # Install dependencies (if not already installed)
# !pip install "paddlepaddle-gpu>=2.5.0" -U
# !pip install -r requirements.txt


  Using cached jiwer-3.0.3-py3-none-any.whl.metadata (2.6 kB)
Using cached jiwer-3.0.3-py3-none-any.whl (21 kB)
  Attempting uninstall: jiwer
    Found existing installation: jiwer 4.0.0
    Uninstalling jiwer-4.0.0:
      Successfully uninstalled jiwer-4.0.0


# Normal Packages

In [21]:
# # Install core packages (HF + metrics). Re-run if Colab restarts.
# !pip -q install -U pip
!pip -q install -U transformers accelerate datasets evaluate jiwer Pillow regex editdistance sentencepiece timm

In [22]:
!pip -q uninstall -y torch torchvision torchaudio
!pip -q install --no-cache-dir torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!python -c "import torch; print(torch.__version__)"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 768.4/768.4 MB 156.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 245.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 253.0 MB/s eta 0:00:00
2.6.0+cu124


In [23]:
!pip -q install safetensors


In [24]:
# Create folders for data and experiments (idempotent)
import os
base = "/content/drive/MyDrive/GothiRead"
subdirs = ["src/data", "src/eval", "src/models", "scripts", "data/train", "data/val", "data/test_public", "exp"]
for sd in subdirs:
    os.makedirs(os.path.join(base, sd), exist_ok=True)
print("Project folders ready at", base)

Project folders ready at /content/drive/MyDrive/GothiRead


Dataset Preparation

In [25]:
import os, glob, json
from pathlib import Path
from PIL import Image
import regex as re

# Adjust base if needed
BASE = "/content/drive/MyDrive/GothiRead"
print("Base:", BASE)

# Make sure python can find our src when you place the repo at BASE
import sys
if BASE not in sys.path:
    sys.path.insert(0, BASE)

from src.data.icdar24 import LineDataset, split_into_chars
from src.eval.metrics import compute_ocr_metrics, compute_font_cer
from src.data.build_vocab import build_char_vocab, save_vocab

Base: /content/drive/MyDrive/GothiRead


In [8]:
!unzip "/content/drive/MyDrive/GothiRead/data/Dataset.zip" -d "/content/dataset"


Streaming output truncated to the last 5000 lines.
 extracting: /content/dataset/valid/single/rotunda/18989.txt  
  inflating: /content/dataset/valid/single/rotunda/138351.font  
  inflating: /content/dataset/valid/single/rotunda/138497.jpg  
  inflating: /content/dataset/valid/single/rotunda/19024.jpg  
 extracting: /content/dataset/valid/single/rotunda/138313.txt  
 extracting: /content/dataset/valid/single/rotunda/138121.txt  
 extracting: /content/dataset/valid/single/rotunda/138014.txt  
  inflating: /content/dataset/valid/single/rotunda/138413.jpg  
 extracting: /content/dataset/valid/single/rotunda/194127.txt  
 extracting: /content/dataset/valid/single/rotunda/138338.txt  
  inflating: /content/dataset/valid/single/rotunda/194110.jpg  
  inflating: /content/dataset/valid/single/rotunda/18983.txt  
  inflating: /content/dataset/valid/single/rotunda/138486.font  
  inflating: /content/dataset/valid/single/rotunda/05410.txt  
  inflating: /content/dataset/valid/single/rotunda/0546

In [9]:
!python /content/drive/MyDrive/GothiRead/scripts/make_test_split.py \
  --root /content \
  --split_ratio 0.10 \
  --include_single True \
  --include_multiple True \
  --move


{
  "root": "/content",
  "train_root": "/content/dataset/train",
  "test_root": "/content/dataset/test",
  "include_single": true,
  "include_multiple": true,
  "split_ratio": 0.1,
  "seed": 42,
  "mode": "move",
  "dry_run": false,
  "leaf_dirs_processed": 17923,
  "triplets_considered": 179223,
  "triplets_selected": 17923
}

Done. Moved 17923 triplet(s) to test.
Test set available at: /content/dataset/test


In [10]:
%cd /content/
!python /content/drive/MyDrive/GothiRead/scripts/build_manifest.py \
  --data-root /content/dataset  \
  --splits train valid test \
  --out-dir manifests

/content
[OK] Wrote manifests/train.csv (163023 rows)
[OK] Wrote manifests/valid.csv (4040 rows)
[OK] Wrote manifests/test.csv (17923 rows)


In [11]:
!python /content/drive/MyDrive/GothiRead/scripts/filter_clean.py \
  --manifests manifests/train.csv manifests/valid.csv manifests/test.csv \
  --out-dir manifests --suffix _clean



[OK] train.csv: kept 161297/163023 (98.94%) -> manifests/train_clean.csv
[OK] valid.csv: kept 3827/4040 (94.73%) -> manifests/valid_clean.csv
[OK] test.csv: kept 17923/17923 (100.00%) -> manifests/test_clean.csv


In [ ]:
!python /content/drive/MyDrive/GothiRead/scripts/debug.py \
  --data-root /content/dataset --split train --n 20


python3: can't open file '/content/drive/MyDrive/GothiRead/scripts/debug.py': [Errno 2] No such file or directory


In [ ]:
# import json, glob

# # runs = glob.glob("/content/runs/*/metrics.json")
# # runs = glob.glob("/content/manifests/donut_sbhavy_donut-base-ocr/metrics.json")
# rows = []
# for path in runs:
#     m = json.load(open(path))
#     rows.append({
#         "run": path.split("/")[-2],
#         "CER": m["CER"],
#         "WER": m["WER"],
#         # "Avg. Latency ms": m.get("timing", {}).get("avg_latency_ms", 0),
#         # "num_beams": m.get("decode", {}).get("num_beams", 1),
#         # "length_penalty": m.get("decode", {}).get("length_penalty", 1.0),
#     })


# import pandas as pd
# df = pd.DataFrame(rows).sort_values("CER")
# print(df)


In [12]:
import csv
from pathlib import Path

def manifest_to_rec_txt(manifest_csv, out_txt):
    ids, imgs, gts = [], [], []
    with open(manifest_csv, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for r in reader:
            if r.get("ok") != "TRUE":
                continue
            img, txt = r["image_path"], r["txt_path"]
            if not (img and txt):
                continue
            gt = Path(txt).read_text(encoding="utf-8").strip()
            imgs.append(img)
            gts.append(gt)
    with open(out_txt, "w", encoding="utf-8") as f:
        for img, gt in zip(imgs, gts):
            f.write(f"{img}\t{gt}\n")

root = "/content"  # adjust if needed

manifest_train = f"{root}/manifests/train_clean.csv"
manifest_val   = f"{root}/manifests/valid_clean.csv"

out_train_txt = "./train_data/gothiread/rec_gt_train.txt"
out_val_txt   = "./train_data/gothiread/rec_gt_val.txt"

Path("./train_data/gothiread").mkdir(parents=True, exist_ok=True)
manifest_to_rec_txt(manifest_train, out_train_txt)
manifest_to_rec_txt(manifest_val, out_val_txt)

print("train:", out_train_txt)
print("val  :", out_val_txt)


train: ./train_data/gothiread/rec_gt_train.txt
val  : ./train_data/gothiread/rec_gt_val.txt


In [13]:
import csv, pathlib

def manifest_to_rec_gt(csv_path, out_txt):
    with open(csv_path, newline="", encoding="utf-8") as f, \
         open(out_txt, "w", encoding="utf-8") as out:
        reader = csv.DictReader(f)
        for row in reader:
            if row.get("ok") not in ("TRUE", "True", "1"):
                continue
            img = row["image_path"]
            txt_path = pathlib.Path(row["txt_path"])
            if not txt_path.is_file():
                continue
            gt = txt_path.read_text(encoding="utf-8").strip()
            if not gt:
                continue
            out.write(f"{img}\t{gt}\n")

manifest_to_rec_gt("/content/manifests/train_clean.csv", "./train_data/gothiread/rec_gt_train.txt")
manifest_to_rec_gt("/content/manifests/valid_clean.csv", "./train_data/gothiread/rec_gt_val.txt")


# TROCR

In [ ]:
# !python /content/drive/MyDrive/GothiRead/scripts/zeroshot/zeroshot_trocr.py \
#         --manifest /content/manifests/valid_clean.csv \
#         --model microsoft/trocr-large-printed \
#         --num_beams 1 \
#         --batch_size 4 \
#         --max_length 128 \
#         # --limit 8000

In [ ]:
# !mkdir -p "/content/drive/MyDrive/manifests_runs_backup"
# !cp -r /content/manifests/runs "/content/drive/MyDrive/manifests_runs_backup"
# print("Copied successfully!")

In [ ]:
# !pip -q install huggingface_hub
# !huggingface-cli login


In [ ]:
# !python - << 'PY'
# import pathlib

# p = pathlib.Path("/content/drive/MyDrive/GothiRead/scripts/finetune_trocr.py")
# text = p.read_text(encoding="utf-8")

# text = text.replace("evaluation_strategy=", "eval_strategy=")

# p.write_text(text, encoding="utf-8")
# print("✔ Patched evaluation_strategy -> eval_strategy")


In [ ]:
# !python /content/drive/MyDrive/GothiRead/scripts/finetune_trocr.py \
#   --train_manifest /content/manifests/train_clean.csv \
#   --val_manifest /content/manifests/valid_clean.csv \
#   --model_name /content/drive/MyDrive/GothiRead/runs/trocr_bs32_a100/checkpoint-2400 \
#   --out_dir /content/drive/MyDrive/GothiRead/runs/trocr_bs40_a100 \
#   --epochs 4 \
#   --train_bs 40 \
#   --eval_bs 16 \
#   --lr 1e-5  # --lr 3e-5


In [ ]:
# !python /content/drive/MyDrive/GothiRead/scripts/finetune_trocr.py \
#   --train_manifest /content/manifests/train_clean.csv \
#   --val_manifest /content/manifests/valid_clean.csv \
#   --model_name microsoft/trocr-base-handwritten \
#   --out_dir /content/drive/MyDrive/GothiRead/runs/trocr_a100_stable \
#   --epochs 3 \
#   --train_bs 54 --grad_accum 1 --eval_bs 8 \
#   --eval_steps 1000 --save_steps 1000 \
#   --max_label_len 128 \
#   --val_eval_limit 1000


# PaddleOCR


In [ ]:
# -------------------------------------------

In [ ]:
# !pip uninstall -y paddlex modelscope torch paddleocr
# !pip install "paddlepaddle-gpu==2.6.2" -f https://www.paddlepaddle.org.cn/whl/linux/mkl/avx/stable.html
# !pip install "paddleocr==2.9.0" opencv-python rapidfuzz jiwer


# # !pip install -U "paddleocr>=3.0.0" "paddlex>=3.0.0"
# # (optional but recommended to avoid older paddlepaddle)
# # For CPU only:
# # !pip install "paddlepaddle==3.0.0b1" -i https://www.paddlepaddle.org.cn/packages/stable/cpu/

# # For GPU (if CUDA 11.8, adjust if needed):
# # !pip install "paddlepaddle-gpu==3.0.0b1" -i https://www.paddlepaddle.org.cn/packages/stable/cu118/


In [ ]:
# !wget https://paddleocr.bj.bcebos.com/PP-OCRv5/rec/en/en_PP-OCRv5_server_rec_infer.tar
# !tar -xf en_PP-OCRv5_server_rec_infer.tar -C /content/


In [14]:
!git clone https://github.com/PaddlePaddle/PaddleOCR.git


Cloning into 'PaddleOCR'...
remote: Enumerating objects: 333264, done.
remote: Counting objects: 100% (101/101), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 333264 (delta 87), reused 73 (delta 73), pack-reused 333163 (from 2)
Receiving objects: 100% (333264/333264), 1.77 GiB | 19.39 MiB/s, done.
Resolving deltas: 100% (264055/264055), done.


In [15]:
%cd /content/PaddleOCR

# # # # 1) Download the PP-OCRv5_server_rec pretrained model
!wget https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv5_server_rec_pretrained.pdparams

# # # # (optional) rename it to something shorter
!mv PP-OCRv5_server_rec_pretrained.pdparams pretrain_ppocrv5_server_rec.pdparams


/content/PaddleOCR
--2026-04-15 17:48:27--  https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv5_server_rec_pretrained.pdparams
Resolving paddle-model-ecology.bj.bcebos.com (paddle-model-ecology.bj.bcebos.com)... 103.235.47.176, 2402:2b40:7000:628:0:ff:b0e8:88da
Connecting to paddle-model-ecology.bj.bcebos.com (paddle-model-ecology.bj.bcebos.com)|103.235.47.176|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 214594738 (205M) [application/octet-stream]
Saving to: ‘PP-OCRv5_server_rec_pretrained.pdparams’

PP-OCRv5_server_rec 100%[===================>] 204.65M  25.9MB/s    in 8.6s    

2026-04-15 17:48:36 (23.8 MB/s) - ‘PP-OCRv5_server_rec_pretrained.pdparams’ saved [214594738/214594738]



In [16]:
import csv, pathlib

def build_char_dict(csv_path, out_path):
    chars = set()
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row.get("ok") not in ("TRUE", "True", "1"):
                continue
            txt_path = pathlib.Path(row["txt_path"])
            if not txt_path.is_file():
                continue
            gt = txt_path.read_text(encoding="utf-8").strip()
            for ch in gt:
                if ch != "\n":
                    chars.add(ch)

    with open(out_path, "w", encoding="utf-8") as out:
        for ch in sorted(chars):
            out.write(ch + "\n")

build_char_dict(
    "/content/manifests/train_clean.csv",
    "/content/PaddleOCR/ppocr/utils/gothi_dict.txt"
)

In [17]:
import csv
from pathlib import Path

def manifest_to_rec_txt(manifest_csv, out_txt):
    ids, imgs, gts = [], [], []
    with open(manifest_csv, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for r in reader:
            if r.get("ok") != "TRUE":
                continue
            img, txt = r["image_path"], r["txt_path"]
            if not (img and txt):
                continue
            gt = Path(txt).read_text(encoding="utf-8").strip()
            imgs.append(img)
            gts.append(gt)
    with open(out_txt, "w", encoding="utf-8") as f:
        for img, gt in zip(imgs, gts):
            f.write(f"{img}\t{gt}\n")

root = "/content"  # adjust if needed

manifest_train = f"{root}/manifests/train_clean.csv"
manifest_val   = f"{root}/manifests/valid_clean.csv"

out_train_txt = "./rec_gt_train.txt"
out_val_txt   = "./rec_gt_val.txt"

Path("./train_data/gothiread").mkdir(parents=True, exist_ok=True)
manifest_to_rec_txt(manifest_train, out_train_txt)
manifest_to_rec_txt(manifest_val, out_val_txt)

print("train:", out_train_txt)
print("val  :", out_val_txt)


train: ./rec_gt_train.txt
val  : ./rec_gt_val.txt


In [ ]:
# !python /content/PaddleOCR/tools/train.py \
#   -c /content/PaddleOCR/PP-OCRv5_gothi_rec.yml \
#   -o Global.use_gpu=True \
#      Global.print_batch_step=50 \
#      Global.checkpoints="./output/PP-OCRv5_server_rec/latest" \
#      Global.eval_batch_step=1000

In [ ]:
# !cp -r /content/PaddleOCR/output/PP-OCRv5_server_rec /content/drive/MyDrive/GothiRead/Backup2_PPOCRv5_server_rec
!cp -r /content/drive/MyDrive/GothiRead/models/PPOCR/Backup2_PPOCRv5_server_rec /content/PaddleOCR/output/PP-OCRv5_server_rec


In [ ]:
!python /content/drive/MyDrive/GothiRead/scripts/patch_dict_164.py


[INFO] Original dict length: 163
[INFO] After de-dup length: 163
[DONE] Patched dict written to: /content/PaddleOCR/ppocr/utils/gothi_dict_164.txt
[DONE] Final dict length: 164


In [ ]:
%cd /content/PaddleOCR

!python tools/eval.py \
  -c /content/PaddleOCR/PP-OCRv5_gothi_rec.yml \
  -o Global.checkpoints=./output/PP-OCRv5_server_rec/best_accuracy

In [ ]:
%cd /content/PaddleOCR
!python tools/export_model.py \
  -c /content/PaddleOCR/PP-OCRv5_gothi_rec.yml \
  -o Global.checkpoints="/content/drive/MyDrive/GothiRead/Results/1. ( REAL ) FineTuned OCR/best_accuracy" \
     Global.save_inference_dir=./inference/PP-OCRv5

/content/PaddleOCR
Skipping import of the encryption module.
W0111 17:55:33.786754  6480 gpu_resources.cc:119] Please NOTE: device: 0, GPU Compute Capability: 7.5, Driver API Version: 12.4, Runtime API Version: 11.8
W0111 17:55:33.804783  6480 gpu_resources.cc:164] device: 0, cuDNN Version: 9.2.
[2026/01/11 17:55:35] ppocr INFO: resume from ./output/PP-OCRv5_server_rec/best_accuracy
[2026/01/11 17:55:35] ppocr INFO: Export inference config file to ./inference/PP-OCRv5/inference.yml
Skipping import of the encryption module
I0111 17:55:38.686044  6480 program_interpreter.cc:212] New Executor is Running.
[2026/01/11 17:55:39] ppocr INFO: inference model is saved to ./inference/PP-OCRv5/inference


In [ ]:
!python /content/drive/MyDrive/GothiRead/scripts/zeroshot_paddleocr.py \
  --manifest /content/manifests/test_clean.csv \
  --rec_model_dir /content/PaddleOCR/inference/PP-OCRv5/ \
  --rec_char_dict_path /content/PaddleOCR/ppocr/utils/gothi_dict.txt \
  --use_gpu

download https://paddleocr.bj.bcebos.com/PP-OCRv4/chinese/ch_PP-OCRv4_det_infer.tar to /root/.paddleocr/whl/det/ch/ch_PP-OCRv4_det_infer/ch_PP-OCRv4_det_infer.tar
100% 4780/4780 [00:00<00:00, 4985.23it/s]
download https://paddleocr.bj.bcebos.com/dygraph_v2.0/ch/ch_ppocr_mobile_v2.0_cls_infer.tar to /root/.paddleocr/whl/cls/ch_ppocr_mobile_v2.0_cls_infer/ch_ppocr_mobile_v2.0_cls_infer.tar
100% 2138/2138 [00:00<00:00, 3431.56it/s]
[2026/03/31 23:36:46] ppocr WARNING: The first GPU is used for inference by default, GPU ID: 0
[2026/03/31 23:36:47] ppocr WARNING: The first GPU is used for inference by default, GPU ID: 0
Saved results to /content/PaddleOCR/runs/ppocrv5_pure/20260331_234243
{
  "CER": 0.01366972429411398,
  "WER": 0.0834514542426707,
  "per_book": {
    "multiple": {
      "edits": 595,
      "chars": 31940,
      "lines": 609,
      "CER": 0.01862867877269881
    },
    "antiqua": {
      "edits": 1433,
      "chars": 132442,
      "lines": 3290,
      "CER": 0.0108198305673

In [ ]:
# !python /content/drive/MyDrive/GothiRead/scripts/prefilter_extract.py \
#   --in /content/manifests/train_clean.csv \
#   --out /content/manifests/train_clean_T40.csv \
#   --max-gt-len 40

# !python /content/drive/MyDrive/GothiRead/scripts/prefilter_extract.py \
#   --in /content/manifests/valid_clean.csv \
#   --out /content/manifests/valid_clean_T40.csv \
#   --max-gt-len 40


[OK] /content/manifests/train_clean_T40.csv: kept 66706/161297 (41.36%) with max_gt_len=40
[OK] /content/manifests/valid_clean_T40.csv: kept 1617/3827 (42.25%) with max_gt_len=40


In [ ]:
!python /content/drive/MyDrive/GothiRead/scripts/extract_align.py \
  --manifest /content/manifests/train_clean.csv \
  --rec_model_dir /content/PaddleOCR/inference/PP-OCRv5 \
  --rec_char_dict_path /content/PaddleOCR/ppocr/utils/gothi_dict.txt \
  --out /content/manifests/train_align.jsonl \
  --rec_image_shape 3,32,320 \
  --batch_size 32 \
  --use_gpu

!python /content/drive/MyDrive/GothiRead/scripts/extract_align.py \
  --manifest /content/manifests/valid_clean.csv \
  --rec_model_dir /content/PaddleOCR/inference/PP-OCRv5 \
  --rec_char_dict_path /content/PaddleOCR/ppocr/utils/gothi_dict.txt \
  --out /content/manifests/valid_align.jsonl \
  --rec_image_shape 3,32,320 \
  --batch_size 32 \
  --use_gpu


[DICT] loaded 163 tokens from infer_cfg in /content/PaddleOCR/inference/PP-OCRv5
--- Running analysis [ir_graph_build_pass]
I0111 23:40:00.655197 96075 executor.cc:187] Old Executor is Running.
--- Running analysis [ir_analysis_pass]
--- Running IR pass [map_op_to_another_pass]
I0111 23:40:00.714521 96075 fuse_pass_base.cc:59] ---  detected 29 subgraphs
--- Running IR pass [is_test_pass]
--- Running IR pass [simplify_with_basic_ops_pass]
--- Running IR pass [delete_quant_dequant_linear_op_pass]
--- Running IR pass [delete_weight_dequant_linear_op_pass]
--- Running IR pass [constant_folding_pass]
I0111 23:40:00.786854 96075 fuse_pass_base.cc:59] ---  detected 3 subgraphs
--- Running IR pass [silu_fuse_pass]
--- Running IR pass [conv_bn_fuse_pass]
I0111 23:40:00.977113 96075 fuse_pass_base.cc:59] ---  detected 86 subgraphs
--- Running IR pass [conv_eltwiseadd_bn_fuse_pass]
--- Running IR pass [embedding_eltwise_layernorm_fuse_pass]
--- Running IR pass [multihead_matmul_fuse_pass_v2]
--- 

In [ ]:
!python /content/drive/MyDrive/GothiRead/scripts/font_vocab.py \
  --align-jsonl /content/manifests/train_align.jsonl \
  --out /content/manifests/font_vocab.json \
  --min-count 5


[OK] wrote /content/manifests/font_vocab.json num_fonts=8 total_graphemes=662488


In [ ]:
import json
from collections import Counter

in_path  = "/content/manifests/train_align.jsonl"
out_path = "/content/manifests/train_align_OK_T40.jsonl"

MAX_GT_LEN = 40      # keep it consistent with your earlier T40 idea
MIN_GT_LEN = 5

kept = 0
drop = Counter()

with open(in_path, "r", encoding="utf-8") as f_in, open(out_path, "w", encoding="utf-8") as f_out:
    for line in f_in:
        ex = json.loads(line)

        ok = ex.get("ok_align", True)  # if field missing, assume ok
        if not ok:
            drop[f"bad_align:{ex.get('reason','UNKNOWN')}"] += 1
            continue

        gt_len = ex.get("gt_units_len", None)
        pred_len = ex.get("pred_units_len", None)

        # If those fields exist, enforce match
        if gt_len is not None and pred_len is not None and int(gt_len) != int(pred_len):
            drop["len_mismatch"] += 1
            continue

        # Keep lengths in a reasonable range (curriculum)
        if gt_len is not None:
            gt_len = int(gt_len)
            if gt_len < MIN_GT_LEN:
                drop["too_short"] += 1
                continue
            if gt_len > MAX_GT_LEN:
                drop["too_long"] += 1
                continue

        f_out.write(json.dumps(ex, ensure_ascii=False) + "\n")
        kept += 1

print("[OK] wrote:", out_path)
print("[STATS] kept:", kept)
print("[STATS] dropped:", sum(drop.values()))
print("[STATS] drop reasons:")
for k, v in drop.most_common(20):
    print(" ", k, v)


FileNotFoundError: [Errno 2] No such file or directory: '/content/manifests/train_align.jsonl'

In [ ]:
!PYTHONPATH=/content/PaddleOCR:$PYTHONPATH \
python /content/drive/MyDrive/GothiRead/scripts/train_font.py \
  --train-align-jsonl /content/manifests/train_align_loose.jsonl \
  --val-align-jsonl /content/manifests/valid_align_loose.jsonl \
  --font-vocab /content/manifests/font_vocab.json \
  --rec-config /content/PaddleOCR/PP-OCRv5_gothi_rec.yml \
  --rec-checkpoint /content/PaddleOCR/output/PP-OCRv5_server_rec/best_accuracy.pdparams \
  --device gpu \
  --pooling meanmax \
  --context none \
  --oversample contains --oversample-ids 3,6 --oversample-mult 2.0 \
  --effective-beta 0.9999 \
  --focal-gamma 1.2 --label-smoothing 0.02 \
  --epochs 10 --batch-size 32 --lr 2e-4 --weight-decay 1e-4 \
  --hidden 384 --dropout 0.10 --grad-clip 1.0 \
  --num-workers 0 \
  --save-best \
  --out-dir runs/font_best_dynamiclr_si_mult2


[INFO] num_fonts=8 pooling=meanmax context=none oversample=contains
[INFO] contains-oversample: boosted_unique=6702/31779 target_mult=2.0 extra=6702 total_indices=38481 boost_ids=[3, 6]
Skipping import of the encryption module.
W0111 23:27:07.738161 92671 gpu_resources.cc:119] Please NOTE: device: 0, GPU Compute Capability: 7.5, Driver API Version: 12.4, Runtime API Version: 11.8
W0111 23:27:07.738914 92671 gpu_resources.cc:164] device: 0, cuDNN Version: 9.2.
/usr/local/lib/python3.12/dist-packages/paddle/nn/layer/layers.py:2084: UserWarning: Skip loading for head.ctc_head.fc.weight. head.ctc_head.fc.weight receives a shape [120, 165], but the expected shape is [120, 164].
  warnings.warn(f"Skip loading for {key}. " + str(err))
/usr/local/lib/python3.12/dist-packages/paddle/nn/layer/layers.py:2084: UserWarning: Skip loading for head.ctc_head.fc.bias. head.ctc_head.fc.bias receives a shape [165], but the expected shape is [164].
  warnings.warn(f"Skip loading for {key}. " + str(err))
/u

In [ ]:
!PYTHONPATH=/content/PaddleOCR:$PYTHONPATH \
  python /content/drive/MyDrive/GothiRead/scripts/font_report.py \
  --val-align-jsonl /content/manifests/valid_align_loose.jsonl \
  --font-vocab /content/manifests/font_vocab.json \
  --rec-config /content/PaddleOCR/PP-OCRv5_gothi_rec.yml \
  --rec-checkpoint /content/PaddleOCR/output/PP-OCRv5_server_rec/best_accuracy.pdparams \
  --font-head /content/PaddleOCR/runs/font_best_dynamiclr_si_mult2/font_head_best.pdparams \
  --pooling meanmax \
  --batch-size 32 \
  --out-dir runs/font_head_meanmax_sqrtinv/val_report_final


Skipping import of the encryption module.
W0111 23:33:58.861883 94511 gpu_resources.cc:119] Please NOTE: device: 0, GPU Compute Capability: 7.5, Driver API Version: 12.4, Runtime API Version: 11.8
W0111 23:33:58.862860 94511 gpu_resources.cc:164] device: 0, cuDNN Version: 9.2.
/usr/local/lib/python3.12/dist-packages/paddle/nn/layer/layers.py:2084: UserWarning: Skip loading for head.ctc_head.fc.weight. head.ctc_head.fc.weight receives a shape [120, 165], but the expected shape is [120, 164].
  warnings.warn(f"Skip loading for {key}. " + str(err))
/usr/local/lib/python3.12/dist-packages/paddle/nn/layer/layers.py:2084: UserWarning: Skip loading for head.ctc_head.fc.bias. head.ctc_head.fc.bias receives a shape [165], but the expected shape is [164].
  warnings.warn(f"Skip loading for {key}. " + str(err))
/usr/local/lib/python3.12/dist-packages/paddle/nn/layer/layers.py:2084: UserWarning: Skip loading for head.gtc_head.embedding.embedding.weight. head.gtc_head.embedding.embedding.weight rec

In [ ]:
!python /content/drive/MyDrive/GothiRead/scripts/find_missing_token.py \
  --manifest /content/manifests/train_clean_T40.csv \
  --dict /content/PaddleOCR/ppocr/utils/gothi_dict.txt \
  --max_rows 50000



Traceback (most recent call last):
  File "/content/drive/MyDrive/GothiRead/scripts/find_missing_token.py", line 20, in <module>
    from units import normalize, strip_ws, glyphs  # must exist in your repo
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
ImportError: cannot import name 'glyphs' from 'units' (/content/drive/MyDrive/GothiRead/scripts/units.py)


# OMNI VS Specialist

## OCR

In [18]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [26]:
%cd /content/PaddleOCR
!python tools/export_model.py \
  -c /content/PaddleOCR/PP-OCRv5_gothi_rec.yml \
  -o Global.checkpoints="/content/drive/MyDrive/GothiRead/Results/1. ( REAL ) FineTuned OCR/best_accuracy" \
     Global.save_inference_dir=./inference/PP-OCRv5

/content/PaddleOCR
Skipping import of the encryption module.
W0415 17:52:52.508661  7015 gpu_resources.cc:119] Please NOTE: device: 0, GPU Compute Capability: 7.5, Driver API Version: 13.0, Runtime API Version: 11.8
W0415 17:52:52.525404  7015 gpu_resources.cc:164] device: 0, cuDNN Version: 9.8.
[2026/04/15 17:53:06] ppocr INFO: resume from /content/drive/MyDrive/GothiRead/Results/1. ( REAL ) FineTuned OCR/best_accuracy
[2026/04/15 17:53:06] ppocr INFO: Export inference config file to ./inference/PP-OCRv5/inference.yml
Skipping import of the encryption module
I0415 17:53:09.207695  7015 program_interpreter.cc:212] New Executor is Running.
[2026/04/15 17:53:09] ppocr INFO: inference model is saved to ./inference/PP-OCRv5/inference


In [ ]:
!python /content/drive/MyDrive/GothiRead/scripts/prepare_paddleocr_specialists.py \
  --train_manifest /content/manifests/train_clean.csv \
  --val_manifest /content/manifests/valid_clean.csv \
  --template_config /content/PaddleOCR/PP-OCRv5_gothi_rec.yml \
  --data_root /content/dataset \
  --out_root /content/drive/MyDrive/GothiRead/Results/paddleocr-specialists \
  --families all_single_plus_multiple \
  --paddleocr_root /content/PaddleOCR


[
  {
    "family": "antiqua",
    "num_train_rows": 29607,
    "num_val_rows": 653,
    "num_train_labels": 29607,
    "num_val_labels": 653,
    "subset_dir": "/content/drive/MyDrive/GothiRead/Results/paddleocr-specialists/antiqua",
    "train_subset_csv": "/content/drive/MyDrive/GothiRead/Results/paddleocr-specialists/antiqua/train_subset.csv",
    "val_subset_csv": "/content/drive/MyDrive/GothiRead/Results/paddleocr-specialists/antiqua/val_subset.csv",
    "train_labels": "/content/drive/MyDrive/GothiRead/Results/paddleocr-specialists/antiqua/rec_gt_train.txt",
    "val_labels": "/content/drive/MyDrive/GothiRead/Results/paddleocr-specialists/antiqua/rec_gt_val.txt",
    "config_path": "/content/drive/MyDrive/GothiRead/Results/paddleocr-specialists/antiqua/PP-OCRv5_gothi_antiqua.yml",
    "train_cmd": "/usr/bin/python3 /content/PaddleOCR/tools/train.py -c /content/drive/MyDrive/GothiRead/Results/paddleocr-specialists/antiqua/PP-OCRv5_gothi_antiqua.yml",
    "eval_cmd": "/usr/bin/pyt

In [27]:
# Then repeat with:
# antiqua (Started)
# bastarda
# fraktur
# gotico-antiqua
# italic
# rotunda
# schwabacher
# textura
# multiple
families = [
    "antiqua", "bastarda", "fraktur", "gotico-antiqua",
    "italic", "rotunda", "schwabacher", "textura", "multiple"
]
families


['antiqua',
 'bastarda',
 'fraktur',
 'gotico-antiqua',
 'italic',
 'rotunda',
 'schwabacher',
 'textura',
 'multiple']

In [38]:
FAMILY = "multiple"   # change this only
BASE = "/content/drive/MyDrive/GothiRead"
PPOCR = "/content/PaddleOCR"

SPEC = f"{BASE}/Results/paddleocr-specialists/{FAMILY}"
CFG = f"{SPEC}/PP-OCRv5_gothi_{FAMILY}.yml"
SUBSET = f"{SPEC}/val_subset.csv"
BEST = f"{SPEC}/output/best_accuracy"
LATEST = f"{SPEC}/output/latest"
INFER = f"{SPEC}/inference"

OMNI_INFER = "/content/PaddleOCR/inference/PP-OCRv5_server_rec"
DICT = "/content/PaddleOCR/ppocr/utils/gothi_dict.txt"

OMNI_OUT = f"{BASE}/Results/ocr-omni-on-{FAMILY}"
SPEC_OUT = f"{BASE}/Results/ocr-specialist-on-{FAMILY}"


In [35]:
%cd $PPOCR
!python -u /content/PaddleOCR/tools/train.py -c $CFG -o Global.checkpoints=$LATEST


/content/PaddleOCR
Skipping import of the encryption module.
[2026/04/15 18:19:41] ppocr INFO: Architecture : 
[2026/04/15 18:19:41] ppocr INFO:     Backbone : 
[2026/04/15 18:19:41] ppocr INFO:         name : PPHGNetV2_B4
[2026/04/15 18:19:41] ppocr INFO:         text_rec : True
[2026/04/15 18:19:41] ppocr INFO:     Head : 
[2026/04/15 18:19:41] ppocr INFO:         head_list : 
[2026/04/15 18:19:41] ppocr INFO:             CTCHead : 
[2026/04/15 18:19:41] ppocr INFO:                 Head : 
[2026/04/15 18:19:41] ppocr INFO:                     fc_decay : 1e-05
[2026/04/15 18:19:41] ppocr INFO:                 Neck : 
[2026/04/15 18:19:41] ppocr INFO:                     depth : 2
[2026/04/15 18:19:41] ppocr INFO:                     dims : 120
[2026/04/15 18:19:41] ppocr INFO:                     hidden_dims : 120
[2026/04/15 18:19:41] ppocr INFO:                     kernel_size : [1, 3]
[2026/04/15 18:19:41] ppocr INFO:                     name : svtr
[2026/04/15 18:19:41] ppocr INFO

In [41]:
%cd $PPOCR
!python -u tools/eval.py -c $CFG -o Global.checkpoints=$LATEST
# !python -u tools/eval.py -c $CFG -o Global.checkpoints=$BEST


/content/PaddleOCR
Skipping import of the encryption module.
[2026/04/15 18:26:11] ppocr INFO: Architecture : 
[2026/04/15 18:26:11] ppocr INFO:     Backbone : 
[2026/04/15 18:26:11] ppocr INFO:         name : PPHGNetV2_B4
[2026/04/15 18:26:11] ppocr INFO:         text_rec : True
[2026/04/15 18:26:11] ppocr INFO:     Head : 
[2026/04/15 18:26:11] ppocr INFO:         head_list : 
[2026/04/15 18:26:11] ppocr INFO:             CTCHead : 
[2026/04/15 18:26:11] ppocr INFO:                 Head : 
[2026/04/15 18:26:11] ppocr INFO:                     fc_decay : 1e-05
[2026/04/15 18:26:11] ppocr INFO:                 Neck : 
[2026/04/15 18:26:11] ppocr INFO:                     depth : 2
[2026/04/15 18:26:11] ppocr INFO:                     dims : 120
[2026/04/15 18:26:11] ppocr INFO:                     hidden_dims : 120
[2026/04/15 18:26:11] ppocr INFO:                     kernel_size : [1, 3]
[2026/04/15 18:26:11] ppocr INFO:                     name : svtr
[2026/04/15 18:26:11] ppocr INFO

In [42]:
%cd $PPOCR
!python -u tools/export_model.py -c $CFG -o Global.checkpoints=$LATEST Global.save_inference_dir=$INFER
# !python -u tools/export_model.py -c $CFG -o Global.checkpoints=$BEST Global.save_inference_dir=$INFER


/content/PaddleOCR
Skipping import of the encryption module.
W0415 18:26:31.354818 15764 gpu_resources.cc:119] Please NOTE: device: 0, GPU Compute Capability: 7.5, Driver API Version: 13.0, Runtime API Version: 11.8
W0415 18:26:31.355676 15764 gpu_resources.cc:164] device: 0, cuDNN Version: 9.8.
[2026/04/15 18:26:32] ppocr INFO: resume from /content/drive/MyDrive/GothiRead/Results/paddleocr-specialists/multiple/output/latest
[2026/04/15 18:26:32] ppocr INFO: Export inference config file to /content/drive/MyDrive/GothiRead/Results/paddleocr-specialists/multiple/inference/inference.yml
Skipping import of the encryption module
I0415 18:26:35.112426 15764 program_interpreter.cc:212] New Executor is Running.
[2026/04/15 18:26:35] ppocr INFO: inference model is saved to /content/drive/MyDrive/GothiRead/Results/paddleocr-specialists/multiple/inference/inference


In [44]:
# !python $BASE/scripts/zeroshot_paddleocr.py \
#   --manifest $SUBSET \
#   --use_gpu \
#   --rec_model_dir $OMNI_INFER \
#   --rec_char_dict_path $DICT \
#   --out_dir $OMNI_OUT

!python /content/drive/MyDrive/GothiRead/scripts/zeroshot_paddleocr.py \
  --manifest $SUBSET \
  --rec_model_dir /content/PaddleOCR/inference/PP-OCRv5/ \
  --rec_char_dict_path /content/PaddleOCR/ppocr/utils/gothi_dict.txt \
  --use_gpu \
  --out_dir $OMNI_OUT




[2026/04/15 18:27:23] ppocr WARNING: The first GPU is used for inference by default, GPU ID: 0
[2026/04/15 18:27:24] ppocr WARNING: The first GPU is used for inference by default, GPU ID: 0
Saved results to /content/drive/MyDrive/GothiRead/Results/ocr-omni-on-multiple
{
  "CER": 0.023327102803738318,
  "WER": 0.12230552952202436,
  "per_book": {
    "multiple": {
      "edits": 312,
      "chars": 13375,
      "lines": 317,
      "CER": 0.023327102803738318
    }
  },
  "per_fontmix": {
    "multiple": {
      "edits": 312,
      "chars": 13375,
      "lines": 317,
      "CER": 0.023327102803738318
    }
  },
  "model": {
    "engine": "PaddleOCR",
    "rec_model_dir": "/content/PaddleOCR/inference/PP-OCRv5/",
    "rec_char_dict_path": "/content/PaddleOCR/ppocr/utils/gothi_dict.txt",
    "det": false,
    "cls": false
  },
  "val_manifest": "/content/drive/MyDrive/GothiRead/Results/paddleocr-specialists/multiple/val_subset.csv",
  "runtime_seconds": 6.737607332999687,
  "avg_latency_ms

In [43]:
!python $BASE/scripts/zeroshot_paddleocr.py \
  --manifest $SUBSET \
  --use_gpu \
  --rec_model_dir $INFER \
  --rec_char_dict_path $DICT \
  --out_dir $SPEC_OUT


[2026/04/15 18:27:00] ppocr WARNING: The first GPU is used for inference by default, GPU ID: 0
[2026/04/15 18:27:01] ppocr WARNING: The first GPU is used for inference by default, GPU ID: 0
Saved results to /content/drive/MyDrive/GothiRead/Results/ocr-specialist-on-multiple
{
  "CER": 0.050542056074766355,
  "WER": 0.25679475164011245,
  "per_book": {
    "multiple": {
      "edits": 676,
      "chars": 13375,
      "lines": 317,
      "CER": 0.050542056074766355
    }
  },
  "per_fontmix": {
    "multiple": {
      "edits": 676,
      "chars": 13375,
      "lines": 317,
      "CER": 0.050542056074766355
    }
  },
  "model": {
    "engine": "PaddleOCR",
    "rec_model_dir": "/content/drive/MyDrive/GothiRead/Results/paddleocr-specialists/multiple/inference",
    "rec_char_dict_path": "/content/PaddleOCR/ppocr/utils/gothi_dict.txt",
    "det": false,
    "cls": false
  },
  "val_manifest": "/content/drive/MyDrive/GothiRead/Results/paddleocr-specialists/multiple/val_subset.csv",
  "runti

In [45]:
import json

omni = json.load(open(f"{OMNI_OUT}/metrics.json", encoding="utf-8"))
spec = json.load(open(f"{SPEC_OUT}/metrics.json", encoding="utf-8"))

print("Family:", FAMILY)
print("Omni CER:", omni["CER"], "WER:", omni["WER"])
print("Spec CER:", spec["CER"], "WER:", spec["WER"])
print("Delta CER:", spec["CER"] - omni["CER"])
print("Delta WER:", spec["WER"] - omni["WER"])


Family: multiple
Omni CER: 0.023327102803738318 WER: 0.12230552952202436
Spec CER: 0.050542056074766355 WER: 0.25679475164011245
Delta CER: 0.027214953271028037
Delta WER: 0.1344892221180881


In [ ]:
exit()

In [ ]:
from google.colab import runtime
runtime.unassign()

# Donut


In [ ]:
# !python /content/drive/MyDrive/GothiRead/scripts/zeroshot/zeroshot_donut.py \
#     --manifest /content/manifests/valid_clean.csv \
#     --model sbhavy/donut-base-ocr \
#     --batch_size 1 \
#     --max_length 96 \
#     --image_size 640 \


# !python /content/drive/MyDrive/GothiRead/scripts/zeroshot/zeroshot_donut.py \
#     --manifest /content/manifests/valid_clean.csv \
#     --model naver-clova-ix/donut-base  \
#     --batch_size 1 \
#     --max_length 96 \
#     --image_size 640 \



# ParSeq

In [ ]:
# !python /content/drive/MyDrive/GothiRead/scripts/zeroshot/zero_shot_parseq.py \
#   --manifest /content/manifests/valid_clean.csv \
#     --model abinet \
#     --batch_size 32 \
#     --max_length 32

# !python /content/drive/MyDrive/GothiRead/scripts/zeroshot/zero_shot_parseq.py \
#   --manifest /content/manifests/valid_clean.csv \
#     --model vitstr  \
#     --batch_size 32 \
#     --max_length 32

# !python /content/drive/MyDrive/GothiRead/scripts/zeroshot/zero_shot_parseq.py \
#   --manifest /content/manifests/valid_clean.csv \
#     --model parseq \
#     --batch_size 32 \
#     --max_length 32 \
#     --image_height 32 \
#     --image_width 128 \
#     --fp16

